# DLR-TomoSAR — Dataset exploration
This notebook inspects how datasets are created, their formats, and where raw data files live inside the repository and the system paths used by training scripts.

Objectives:
- Load and inspect `core/dataset.py` dataset classes.
- Search the repository for `.rat`, `.npy`, and common raw data files.
- Attempt a safe, non-destructive load of a sample file if available.

In [ ]:
# Basic imports and repository root
import os
import sys
from pathlib import Path
repo_root = Path('/ste/rnd/User/vice_vi/DLR-TomoSAR')
print('Repository root set to', repo_root)
# Ensure repo root is on sys.path so we can import `core`
sys.path.insert(0, str(repo_root))

In [ ]:
# Inspect the dataset module and key classes
try:
    from core import dataset as ds
    print('Imported core.dataset as ds')
    print('
Available names in core.dataset:')
    print([n for n in dir(ds) if not n.startswith('_')])
    # Show brief docstrings for main classes
    for name in ('SLCPatchDataset', 'FeatureMapDataset', 'ParameterEstimationDataset', 'TensorPairDataset', 'DataPipeline', 'Representation'):
        if hasattr(ds, name):
            obj = getattr(ds, name)
            print(f'
{name}:')
            print(obj.__doc__ or obj.__name__)
except Exception as e:
    print('Could not import core.dataset:', e)

In [ ]:
# Search the repository for common raw/feature file types (.rat, .npy, .h5)
extensions = ('.rat', '.npy', '.h5', '.npz')
found = {ext: [] for ext in extensions}
for root, dirs, files in os.walk(repo_root):
    for f in files:
        for ext in extensions:
            if f.lower().endswith(ext):
                found[ext].append(Path(root) / f)
# Print a few examples per extension
for ext, paths in found.items():
    print(f'\n{ext} — found {len(paths)} files. Examples:')
    for p in paths[:10]:
        print(' -', p)

# Also show the external path used by training scripts (if present)
external_path = Path('/ste/rnd/User/sera_se')
print('
External training path referenced in scripts:', external_path)
if external_path.exists():
    print('External path exists; listing a few files:')
    for i, p in enumerate(sorted(external_path.iterdir())):
        print(' -', p)
        if i >= 20:
            break
else:
    print('External path does not exist on this machine or is not accessible from the notebook kernel.')

In [ ]:
# Attempt a safe inspection of a .rat file using STEtools.rat reader if available
from pprint import pprint
sample_rat = None
# Prefer a .rat inside the repo if found, otherwise show example from external path
if found.get('.rat'):
    sample_rat = found['.rat'][0]
elif external_path.exists():
    # try common names used in training script
    for name in ('Elsa96x96ref0s.rat', 'Ana96x96ref0s.rat', 'Nani_small_96x96.rat'):
        p = external_path / name
        if p.exists():
            sample_rat = p
            break

print('Sample .rat chosen for inspection:', sample_rat)
if sample_rat is None:
    print('No .rat file found to inspect. If your raw data is outside this repo, update the external path or copy a sample file into the notebook environment.')
else:
    try:
        from STEtools.ste_io import rrat
        print('rrat available; attempting a lightweight header read...')
        # rrat() loads the full array; avoid printing large arrays. Instead, print shape and dtype.
        arr = rrat(str(sample_rat))
        print('rrat loaded array shape:', getattr(arr, 'shape', None))
        print('dtype:', getattr(arr, 'dtype', None))
        # If array is large, show a small slice summary
        if hasattr(arr, 'shape') and arr.size > 0:
            print('Summary statistics for a tiny sample slice:')
            import numpy as np
            s = arr.reshape(-1)[:1000]  # sample up to first 1000 elements
            print('min, max, mean, std =', float(np.min(s)), float(np.max(s)), float(np.mean(s)), float(np.std(s)))
    except Exception as e:
        print('Could not import or read using STEtools.ste_io.rrat:', e)
        print('As an alternative, try `FeatureMapDataset` for .npy feature files inside the repo.')

In [ ]:
# Inspect .npy feature files (if any) using a safe small-load approach
import numpy as np
npy_paths = found.get('.npy', [])
print('Found .npy files:', len(npy_paths))
for p in npy_paths[:10]:
    try:
        print('Loading', p)
        a = np.load(p, mmap_mode='r')
        print(' - shape:', a.shape, 'dtype:', a.dtype)
        # show min/max on a small sample to avoid heavy IO
        flat = a.reshape(-1)
        sample = flat[:1000] if flat.size else flat
        if sample.size:
            print(' - min/max/mean (sample):', float(sample.min()), float(sample.max()), float(sample.mean()))
    except Exception as e:
        print(' - could not read', p, '->', e)

**Summary / next steps**
- The repository exposes dataset classes in `core/dataset.py`: `SLCPatchDataset`, `FeatureMapDataset`, `ParameterEstimationDataset`, `TensorPairDataset`, and `DataPipeline`.
- `SLCPatchDataset` reads `.rat` files via `STEtools.ste_io.rrat` and flattens patch grids into (N, P, H, W) style arrays.
- `FeatureMapDataset` expects directories of `.npy` feature files and stacks them along channel axis.
- If the notebook kernel can access external data paths (e.g., `/ste/rnd/User/sera_se/` used in training scripts), the code above will list and perform lightweight inspections.

If you want, I can now: (a) run this notebook here to produce outputs, (b) adapt it to load a specific sample file, or (c) add cells showing how to create a `DataPipeline` and fetch a single training batch. Which do you prefer?